In [1]:
"""
IC2024 dataset module to run MedSigLip Embeddings
"""
import os
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import argparse
import sys
# Get absolute path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)
from utils.f_utils import load_config, load_models, load_dataset
from PIL import Image


# --------------
# Classe Dataset
# --------------

class ImageCLEF24_Dataset(Dataset):
    def __init__(self, split, processor, max_length=64):
        """
        split: split do dataset (ex: ds["train"])
        processor: processor do MedSigLip
        max_length: tamanho máximo do texto
        """
        self.dataset = split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Carrega a imagem apenas agora
        image = Image.open(sample["image_path"]).convert('RGB')
        text = sample["caption"]

        # Usa o processor do modelo
        # texto (caption) + imagem -> embedding multimodal do MedSigLip ?
        encoding = self.processor(
            text=text,
            images=image,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Remove batch dimension criada pelo return_tensors="pt"
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        return encoding


# --------------
# Calcula Embeddings
# --------------

In [2]:
import torch
torch.cuda.empty_cache()

In [3]:

config = load_config("../configs/imageclef2026_emb_configs.yaml")
processor, model, device = load_models(config, device=config['device'])

print("Loading Dataset ")
ds = load_dataset(config)

dataset = ImageCLEF24_Dataset(
    split=ds,
    processor=processor,
    max_length=64
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    pin_memory=True
)



The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Loading Dataset 


In [4]:
batch = {k: v.to(device) for k, v in next(iter(dataloader)).items()}

In [5]:
image_embeds_old = model.get_image_features(
        pixel_values=batch["pixel_values"]
    )

text_embeds_old = model.get_text_features(
    input_ids=batch["input_ids"]
)


In [6]:
image_embeds_old

BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.2872, -0.1459,  0.0390,  ..., -0.1010,  0.5056, -0.0684],
         [-0.1834, -0.9177,  0.3355,  ...,  0.1094,  0.7672, -0.5063],
         [ 0.4802, -0.1182,  0.6433,  ..., -0.5499,  0.6789, -0.1868],
         ...,
         [ 0.1677, -0.4340,  0.0578,  ...,  0.3264,  0.1010, -0.2228],
         [ 0.0090, -0.5055,  0.6822,  ..., -0.5676,  0.3798, -0.3203],
         [ 0.3497, -0.2640,  0.0975,  ...,  0.1909,  0.3521, -0.3758]],

        [[ 0.6452, -0.0631,  0.6451,  ...,  0.0859,  0.2207,  0.3065],
         [-0.2320, -0.4295,  0.8344,  ..., -0.2519,  1.1401,  0.1519],
         [ 0.3645,  0.1714,  0.2568,  ..., -0.0556,  0.2469,  0.2903],
         ...,
         [ 0.5518,  0.0623, -0.0134,  ..., -0.0372, -0.0079,  0.1631],
         [ 0.2255,  0.1957, -0.0178,  ..., -0.1115, -0.1859,  0.1929],
         [ 0.6163, -0.0237,  0.2921,  ..., -0.0554,  0.3567,  0.1709]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>), pooler_out

In [7]:
image_embeds_old = image_embeds_old['pooler_output']

image_embeds = image_embeds_old / image_embeds_old.norm(dim=-1, keepdim=True)

In [5]:
train_emb = torch.load("../artifacts/embeddings/IC2024_medsiglip_train_image_embeddings.pt").numpy()
train_captions = torch.load("../artifacts/embeddings/IC2024_medsiglip_train_captions.pt")

In [7]:
train_emb

array([[-0.00877565,  0.01093615,  0.01460912, ..., -0.00175851,
         0.06190941, -0.03780896],
       [ 0.04382763, -0.00310173,  0.00734794, ...,  0.01207565,
         0.02902102, -0.01495183],
       [-0.02099044,  0.02116433, -0.02201966, ...,  0.01648863,
         0.01743797, -0.04344815],
       ...,
       [ 0.01755939, -0.02507309,  0.00097105, ...,  0.01447324,
         0.06572567, -0.01865052],
       [ 0.02841968, -0.00849806, -0.00611143, ..., -0.01393911,
         0.00120339, -0.01557527],
       [-0.01387829,  0.02283915, -0.00617081, ...,  0.03843228,
         0.01038149, -0.005943  ]], dtype=float32)

In [4]:
image_embeds_old

(tensor([[[ 0.2875, -0.1467,  0.0390,  ..., -0.1009,  0.5056, -0.0685],
          [-0.1826, -0.9162,  0.3353,  ...,  0.1094,  0.7680, -0.5031],
          [ 0.4801, -0.1184,  0.6432,  ..., -0.5510,  0.6803, -0.1867],
          ...,
          [ 0.1674, -0.4344,  0.0578,  ...,  0.3261,  0.1020, -0.2232],
          [ 0.0090, -0.5057,  0.6818,  ..., -0.5681,  0.3806, -0.3203],
          [ 0.3485, -0.2651,  0.0985,  ...,  0.1914,  0.3526, -0.3762]]],
        device='cuda:0', grad_fn=<NativeLayerNormBackward0>),
 tensor([[-0.0608,  0.0757,  0.1011,  ..., -0.0126,  0.4287, -0.2623]],
        device='cuda:0', grad_fn=<SelectBackward0>))

In [14]:
print(model.__class__)

<class 'transformers.models.siglip.modeling_siglip.SiglipModel'>


In [15]:
import transformers
print(transformers.__version__)

5.2.0


In [12]:
text_embeds

BaseModelOutputWithPooling(last_hidden_state=tensor([[[-0.4809,  2.1400,  1.3087,  ...,  0.4018, -1.9341, -3.2021],
         [ 0.3296, -0.2725, -0.2671,  ..., -0.7298,  0.9657, -1.3528],
         [-1.8965,  1.4593, -1.6124,  ...,  1.1742, -0.4809, -1.2341],
         ...,
         [ 0.0252,  1.1518, -0.0905,  ..., -0.7575,  0.6743, -2.1341],
         [-0.3349,  0.7150, -0.1474,  ..., -1.9227,  1.0026,  0.3731],
         [-0.5428,  0.7710, -0.3532,  ..., -1.9500,  0.8913, -0.2268]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[ 0.7847,  2.9182, -0.3157,  ..., -0.0730,  0.5320, -0.7204]],
       device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)